# Pythia BEACON Structural Tasks

This notebook demonstrates toy examples for **SSP**, **CMP**, **DMP**, and **SSI** using synthetic stem-loop sequences.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, str(Path('..').resolve().parent))

from pythia.configs import SSPModelConfig, CMPModelConfig, DMPModelConfig, SSIModelConfig
from pythia.models.ssp import PythiaSSP, compute_ssp_metrics
from pythia.models.cmp import PythiaCMP, topL_precision
from pythia.models.dmp import PythiaDMP, compute_dmp_metrics
from pythia.models.ssi import PythiaSSI

matplotlib.use('Agg')
sns.set_theme(style='whitegrid', context='notebook')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Toy Data Generation

Sequences have length 20 nt with a 4-bp stem + 6-nt loop structure.
- Stem pairing: pos 0-3 ↔ pos 19-16 (Watson-Crick)
- Additional 4 random flanking nts on each side of the stem
- Contact matrix has 1s at stem pairs

In [ ]:
NUCS = ['A', 'C', 'G', 'U']
RNA_MAP = {'A': 0, 'C': 1, 'G': 2, 'U': 3}
RC_MAP = {'A': 'U', 'U': 'A', 'C': 'G', 'G': 'C'}
SEQ_LEN = 20
STEM_LEN = 4
LOOP_LEN = 6
FLANK_LEN = 3  # random flanks on each side


def rand_rna(n):
    return [random.choice(NUCS) for _ in range(n)]


def make_stem_loop_20():
    """20-nt sequence: 3-flank + 4-stem + 6-loop + 4-rc-stem + 3-flank."""
    left_flank = rand_rna(FLANK_LEN)
    stem = rand_rna(STEM_LEN)
    loop = rand_rna(LOOP_LEN)
    rc_stem = [RC_MAP[n] for n in reversed(stem)]
    right_flank = rand_rna(FLANK_LEN)
    return left_flank + stem + loop + rc_stem + right_flank


def make_contact_matrix(seq_list):
    """Build L x L contact matrix for a stem-loop-20 sequence list."""
    L = len(seq_list)
    mat = np.zeros((L, L), dtype=np.float32)
    # Stem pairing: positions 3..6 pair with 16..13 (reversed)
    for k in range(STEM_LEN):
        i = FLANK_LEN + k
        j = L - 1 - FLANK_LEN - k
        if 0 <= i < L and 0 <= j < L and i != j:
            mat[i, j] = 1.0
            mat[j, i] = 1.0
    return mat


def one_hot_batch(seqs, length=SEQ_LEN):
    B = len(seqs)
    arr = np.zeros((B, 4, length), dtype=np.float32)
    for bi, seq in enumerate(seqs):
        for i, ch in enumerate(seq[:length]):
            idx = RNA_MAP.get(ch)
            if idx is not None:
                arr[bi, idx, i] = 1.0
    return arr


N = 200
raw_seqs = [make_stem_loop_20() for _ in range(N)]
sequences_str = [''.join(s) for s in raw_seqs]
contact_mats = np.stack([make_contact_matrix(s) for s in raw_seqs])

X_onehot = one_hot_batch(raw_seqs)  # (N, 4, 20)

print(f'Generated {N} toy sequences of length {SEQ_LEN}')
print(f'Example: {sequences_str[0]}')
print(f'Contact matrix shape: {contact_mats.shape}')
print(f'Contacts per sequence: {contact_mats[0].sum() / 2:.0f} pairs')

## 2. Visualise a Stem-Loop Contact Map

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(contact_mats[0], cmap='Blues', vmin=0, vmax=1)
ax.set_title('Toy Stem-Loop Contact Map (L=20)', fontweight='bold')
ax.set_xlabel('Position j', fontweight='bold')
ax.set_ylabel('Position i', fontweight='bold')
plt.tight_layout()
plt.savefig('ssp_toy_contact_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ssp_toy_contact_map.png')

## 3. Secondary Structure Prediction (SSP) — Forward Pass Demo

In [ ]:
ssp_cfg = SSPModelConfig(
    dil_start=2,
    dil_end=8,
    bulge_size=1,
    hidden_dim=64,
    num_res_layers=2,
    dropout=0.1,
    arm1_widths=(32, 32, 32),
    arm2_widths=(64, 32, 32),
    dist_enc_dim=16,
)
ssp_model = PythiaSSP(ssp_cfg).to(device)

x_demo = torch.from_numpy(X_onehot[:4]).to(device)
with torch.no_grad():
    logits_demo = ssp_model(x_demo)

print(f'SSP output logits shape: {logits_demo.shape}')  # (4, 20, 20)
probs_demo = torch.sigmoid(logits_demo).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(contact_mats[0], cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Ground Truth', fontweight='bold')
axes[0].set_xlabel('Position j', fontweight='bold')
axes[0].set_ylabel('Position i', fontweight='bold')

axes[1].imshow(probs_demo[0], cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('PythiaSSP Prediction (untrained)', fontweight='bold')
axes[1].set_xlabel('Position j', fontweight='bold')
axes[1].set_ylabel('Position i', fontweight='bold')

fig.tight_layout()
plt.savefig('ssp_prediction_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ssp_prediction_demo.png')

## 4. Quick SSP Training on Toy Data

In [ ]:
X_t = torch.from_numpy(X_onehot[:160])
y_t = torch.from_numpy(contact_mats[:160])
X_v = torch.from_numpy(X_onehot[160:])
y_v = torch.from_numpy(contact_mats[160:])

criterion_ssp = nn.BCEWithLogitsLoss()
opt_ssp = torch.optim.Adam(ssp_model.parameters(), lr=3e-4)

ssp_train_losses, ssp_val_f1s = [], []

for ep in range(1, 16):
    ssp_model.train()
    opt_ssp.zero_grad()
    logits = ssp_model(X_t.to(device))
    L = y_t.shape[1]
    if logits.shape[1] > L:
        logits = logits[:, :L, :L]
    loss = criterion_ssp(logits, y_t.to(device))
    loss.backward()
    opt_ssp.step()
    ssp_train_losses.append(float(loss))

    ssp_model.eval()
    with torch.no_grad():
        logits_v = ssp_model(X_v.to(device))
        if logits_v.shape[1] > L:
            logits_v = logits_v[:, :L, :L]
        labels_v = y_v.to(device)
        mask_v = labels_v >= 0
        m = compute_ssp_metrics(logits_v, labels_v, mask_v)
    ssp_val_f1s.append(m['f1'])
    print(f'SSP ep {ep:02d} | loss={ssp_train_losses[-1]:.4f} | val_f1={m["f1"]:.4f}')

print('SSP toy training done.')

## 5. Plot SSP Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_ssp = range(1, len(ssp_train_losses) + 1)
axes[0].plot(epochs_ssp, ssp_train_losses, lw=2, color='steelblue')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('BCE Loss', fontweight='bold')
axes[0].set_title('SSP Training Loss', fontweight='bold')

axes[1].plot(epochs_ssp, ssp_val_f1s, lw=2, color='tomato')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('F1 Score', fontweight='bold')
axes[1].set_title('SSP Validation F1', fontweight='bold')

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
plt.savefig('ssp_toy_training.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ssp_toy_training.png')

## 6. CMP Model Forward Pass Demo

In [ ]:
cmp_cfg = CMPModelConfig(
    dil_start=2,
    dil_end=8,
    bulge_size=1,
    hidden_dim=64,
    num_res_layers=2,
    dropout=0.1,
    arm1_widths=(32, 32, 32),
    arm2_widths=(64, 32, 32),
    dist_enc_dim=16,
    min_separation=4,  # smaller for toy data
)
cmp_model = PythiaCMP(cmp_cfg).to(device)

with torch.no_grad():
    cmp_logits = cmp_model(x_demo)
print(f'CMP output shape: {cmp_logits.shape}')

## 7. DMP Model Forward Pass Demo

In [ ]:
dmp_cfg = DMPModelConfig(
    dil_start=2,
    dil_end=8,
    bulge_size=1,
    hidden_dim=64,
    num_res_layers=2,
    dropout=0.1,
    arm1_widths=(32, 32, 32),
    arm2_widths=(64, 32, 32),
    dist_enc_dim=16,
)
dmp_model = PythiaDMP(dmp_cfg).to(device)

with torch.no_grad():
    dmp_preds = dmp_model(x_demo)
print(f'DMP output shape: {dmp_preds.shape}')  # (4, 20, 20), values in [0,1]
print(f'DMP output range: [{dmp_preds.min().item():.4f}, {dmp_preds.max().item():.4f}]')

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(dmp_preds[0].cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
ax.set_title('PythiaDMP Predicted Distance Map (untrained)', fontweight='bold')
ax.set_xlabel('Position j', fontweight='bold')
ax.set_ylabel('Position i', fontweight='bold')
plt.colorbar(ax.images[0], ax=ax, label='Normalised Distance')
plt.tight_layout()
plt.savefig('dmp_prediction_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: dmp_prediction_demo.png')

## 8. SSI Model Forward Pass Demo

In [ ]:
ssi_cfg = SSIModelConfig(
    dil_start=2,
    dil_end=8,
    bulge_size=1,
    hidden_dim=64,
    num_res_layers=2,
    dropout=0.1,
    arm1_widths=(32, 32, 32),
    arm2_widths=(64, 32, 32),
)
ssi_model = PythiaSSI(ssi_cfg).to(device)

# PythiaSSI conditions on the observed (partially-missing) structural scores,
# so the forward pass takes the one-hot sequence plus an observed-score
# channel and a known-mask channel.
obs_normed_demo = torch.zeros(x_demo.shape[0], x_demo.shape[2], device=device)
obs_known_demo = torch.zeros(x_demo.shape[0], x_demo.shape[2], dtype=torch.bool, device=device)

with torch.no_grad():
    ssi_preds = ssi_model(x_demo, obs_normed_demo, obs_known_demo)  # (4, 20)
print(f'SSI output shape: {ssi_preds.shape}')

# Simulate ground-truth structural scores (SHAPE-like values in [0, 2])
struct_scores_gt = np.random.uniform(0, 2, (N, SEQ_LEN)).astype(np.float32)
# Introduce 30% missing values (set to -1)
missing_mask = np.random.rand(N, SEQ_LEN) < 0.3
struct_scores_obs = struct_scores_gt.copy()
struct_scores_obs[missing_mask] = -1.0

print(f'Structural scores shape: {struct_scores_gt.shape}')
print(f'Missing fraction: {missing_mask.mean():.2f}')

## 9. Architecture Summary

In [ ]:
def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, frozen


models = {
    'PythiaSSP': ssp_model,
    'PythiaCMP': cmp_model,
    'PythiaDMP': dmp_model,
    'PythiaSSI': ssi_model,
}

rows = []
for name, m in models.items():
    tr, fr = count_params(m)
    rows.append({'Model': name, 'Trainable': tr, 'Frozen': fr, 'Total': tr + fr})

import pandas as pd
df_params = pd.DataFrame(rows)
print(df_params.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_params))
ax.bar(x, df_params['Trainable'], label='Trainable', color='steelblue')
ax.bar(x, df_params['Frozen'], bottom=df_params['Trainable'], label='Frozen', color='salmon')
ax.set_xticks(x)
ax.set_xticklabels(df_params['Model'])
ax.set_xlabel('Model', fontweight='bold')
ax.set_ylabel('Parameter Count', fontweight='bold')
ax.set_title('Pythia BEACON Model Parameter Counts', fontweight='bold')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('beacon_model_params.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: beacon_model_params.png')